# Pneumodetect — Pneumonia Detection from Chest X-ray Images

**Authors:** Asulkhan Aidarkhanov, Ravil Serikuly · **Group:** BDA-2401
**Course:** Machine Learning Algorithms, Astana IT University · **Instructor:** Assiya Karatay

## README

Pneumodetect is a decision-support prototype that estimates the probability of pneumonia on a frontal
(PA/AP) chest X-ray. It is a course project, **not a medical device**.

### Data (≈100k images, everything is fetched through APIs — nothing is uploaded by hand)

| Source | What we take | Access in this notebook |
|---|---|---|
| NIH ChestX-ray14 (Wang et al., 2017) | 112,120 frontal X-rays of 30,805 patients + metadata | Kaggle API, dataset `nih-chest-xrays/data` |
| RSNA Pneumonia Detection Challenge (Shih et al., 2019) | radiologist labels for 26,684 of the NIH images (label CSV only, no images) | Kaggle API, competition `rsna-pneumonia-detection-challenge` |
| RSNA → NIH mapping (RSNA, 2018) | links every RSNA image ID to its NIH file name | public HTTPS endpoint (JSON) |
| Kermany et al. (2018) | 5,856 paediatric X-rays, Normal / Bacterial / Viral | Kaggle API, dataset `paultimothymooney/chest-xray-pneumonia` |

RSNA images *are* NIH images, so we never use them twice: RSNA contributes only its (more reliable) labels.

### How to run

**Option A — Kaggle Notebooks (recommended: datasets are attached by the API without downloading, free GPU).**
1. Sign in to Kaggle and press *Join / I understand and accept* on the rules page of
   https://www.kaggle.com/competitions/rsna-pneumonia-detection-challenge (needed once, otherwise the API refuses the label file).
2. *Create → New Notebook → File → Import Notebook* and select this `.ipynb`.
3. *Settings*: Accelerator **GPU T4 ×2** (or P100), Internet **On**.
4. *Run All*.

**Option B — Google Colab.**
1. Kaggle → *Settings → API → Create New Token*; accept the RSNA competition rules (step A1).
2. In Colab open *Secrets* (key icon) and add `KAGGLE_USERNAME` and `KAGGLE_KEY`, enable notebook access.
   Never paste the key into a cell — it would end up in the exported PDF.
3. *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*.
4. Needs ~50 GB of free disk: the NIH archive (42 GB) is read **directly from the zip without extracting**
   once into a 224×224 cache (~5.6 GB); set `DELETE_NIH_ZIP_AFTER_CACHE = True` to free the space afterwards.

**Option C — local machine.** `pip install kagglehub tensorflow scikit-learn pandas pillow matplotlib tqdm requests`,
put your token in `~/.kaggle/kaggle.json`, run all cells (needs ~90 GB free for the extracted NIH data).

### Notebook structure (requirement IDs from the proposal)

1. Environment and GPU check — D4
2. Data acquisition through APIs — D1
3. Labels (variant *b*: radiologist-verified pneumonia vs. no pneumonia)
4. Train / validation / test split without patient leakage — D3
5. Image preprocessing to 224×224 — D2
6. Exploratory data analysis
7. Input pipeline, augmentation and class weights — D7, D8
8. Evaluation protocol: metrics on validation, NIH test and Kermany test — D10
9. Classical baselines: logistic regression and linear SVM on pixels and on frozen ResNet50 features — D11

## 1. Environment and GPU check (D4)

In [ ]:
import subprocess, sys

# kagglehub is the official Kaggle API client for Python; Colab and Kaggle ship it, we only make sure it is recent.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub>=0.3.4"], check=False)

In [ ]:
import io, json, os, random, re, threading, time, zipfile
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import tensorflow as tf
from PIL import Image
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_class_weight
from tqdm.auto import tqdm

print("Python     :", sys.version.split()[0])
print("TensorFlow :", tf.__version__)
print("kagglehub  :", kagglehub.__version__)

In [ ]:
# ---- configuration ---------------------------------------------------------------------------------------------
SEED = 42
IMG_SIZE = 224            # D2: every image is resized to 224x224
BATCH_SIZE = 32
VAL_FRACTION = 0.15       # D3: validation split carved out of the training data (grouped by patient)

# Variant (b): positives are only radiologist-verified pneumonia (RSNA "Lung Opacity") or an explicit NIH "Pneumonia"
# label. NEGATIVE_POLICY decides which images count as negatives:
#   "all_non_pneumonia" - every image without pneumonia (No Finding + other diseases)  -> ~100k images
#   "normal_only"       - only images with no finding at all                           -> ~70k images
NEGATIVE_POLICY = os.environ.get("PNEUMODETECT_NEGATIVES", "all_non_pneumonia")

NIH_HANDLE = "nih-chest-xrays/data"
KERMANY_HANDLE = "paultimothymooney/chest-xray-pneumonia"
RSNA_COMPETITION = "rsna-pneumonia-detection-challenge"
RSNA_MAPPING_URL = ("https://s3.amazonaws.com/east1.public.rsna.org/AI/2018/"
                    "pneumonia-challenge-dataset-mappings_2018.json")

IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# "zip": read NIH images straight from the downloaded archive (Colab: avoids extracting 42 GB)
# "dir": use the folder returned by kagglehub (Kaggle Notebooks: attached, nothing is downloaded)
NIH_ACCESS = os.environ.get("PNEUMODETECT_NIH_ACCESS", "zip" if IN_COLAB else "dir")
DELETE_NIH_ZIP_AFTER_CACHE = False   # True frees 42 GB, but a second "Run all" would download it again

WORK_DIR = Path("/kaggle/working") if IN_KAGGLE else Path("/content") if IN_COLAB else Path.cwd()
CACHE_DIR = Path(os.environ.get("PNEUMODETECT_CACHE",
                                "/content/pneumodetect_cache" if IN_COLAB else "/tmp/pneumodetect_cache"))
CACHE_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print(f"Platform: {'Kaggle' if IN_KAGGLE else 'Colab' if IN_COLAB else 'local'} | NIH access: {NIH_ACCESS} | "
      f"negatives: {NEGATIVE_POLICY}")
print("Work dir :", WORK_DIR)
print("Cache dir:", CACHE_DIR)

In [ ]:
# D4: training a CNN on ~100k images on a CPU is far too slow, so we check for a GPU before doing anything heavy.
gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus)
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
        print("  ->", tf.config.experimental.get_device_details(gpu).get("device_name", gpu.name))
else:
    print("WARNING: no GPU found. Data preparation works on CPU, but switch the runtime to a GPU before training.")

## 2. Data acquisition through APIs (D1)

All data come from the Kaggle API (`kagglehub` / `kaggle` CLI) and one public HTTPS endpoint.
Credentials are read from Colab *Secrets* or `~/.kaggle/kaggle.json` and are never printed.

In [ ]:
def setup_kaggle_credentials():
    # Make Kaggle credentials available to kagglehub and the kaggle CLI without printing them.
    if IN_KAGGLE:
        print("Running inside Kaggle: the API is already authenticated.")
        return
    has_env = os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")
    has_file = (Path.home() / ".kaggle" / "kaggle.json").exists()
    if not (has_env or has_file or os.environ.get("KAGGLE_API_TOKEN")) and IN_COLAB:
        from google.colab import userdata
        try:
            for key in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
                os.environ[key] = userdata.get(key)
        except Exception as err:
            raise RuntimeError("Add KAGGLE_USERNAME and KAGGLE_KEY to Colab Secrets (see README).") from err
    print("Kaggle credentials configured (values hidden).")


setup_kaggle_credentials()

In [ ]:
class DirSource:
    # Dataset files that are available as a folder (Kaggle mount or kagglehub download).
    def __init__(self, root):
        self.root = Path(root)
        self.files = sorted(str(p) for p in self.root.rglob("*") if p.is_file())

    def read(self, key):
        with open(key, "rb") as f:
            return f.read()


class ZipSource:
    # Dataset files read straight from a zip archive, without extracting it (thread-safe).
    def __init__(self, zip_path):
        self.zip_path = str(zip_path)
        self._local = threading.local()
        with zipfile.ZipFile(self.zip_path) as zf:
            self.files = sorted(n for n in zf.namelist() if not n.endswith("/"))

    def read(self, key):
        if not hasattr(self._local, "zf"):
            self._local.zf = zipfile.ZipFile(self.zip_path)
        return self._local.zf.read(key)


def find_file(source, filename):
    # Return the key of the (shallowest) file in `source` whose base name is `filename`.
    hits = [k for k in source.files if Path(k).name == filename and "__MACOSX" not in k]
    if not hits:
        raise FileNotFoundError(f"{filename} not found in {getattr(source, 'root', getattr(source, 'zip_path', ''))}")
    return min(hits, key=lambda k: (k.count("/"), k))


def read_csv_from(path, filename):
    # kagglehub may return a folder, the file itself or a zip with the file inside; handle all three.
    path = Path(path)
    if path.is_dir():
        source = DirSource(path)
    elif zipfile.is_zipfile(path):
        source = ZipSource(path)
    else:
        return pd.read_csv(path)
    names = [Path(k).name for k in source.files]
    key = find_file(source, filename) if filename in names else find_file(source, filename + ".zip")
    data = source.read(key)
    if key.endswith(".zip"):
        with zipfile.ZipFile(io.BytesIO(data)) as inner:
            data = inner.read(next(n for n in inner.namelist() if n.endswith(filename)))
    return pd.read_csv(io.BytesIO(data))

In [ ]:
def get_nih_source():
    if NIH_ACCESS == "zip":
        zip_dir = CACHE_DIR / "nih_zip"
        zip_dir.mkdir(exist_ok=True)
        if not list(zip_dir.glob("*.zip")):
            print("Downloading the NIH archive through the Kaggle API (~42 GB, ~10-20 min on Colab)...")
            subprocess.run(["kaggle", "datasets", "download", "-d", NIH_HANDLE, "-p", str(zip_dir)], check=True)
        return ZipSource(next(zip_dir.glob("*.zip")))
    return DirSource(kagglehub.dataset_download(NIH_HANDLE))


t0 = time.time()
nih_src = get_nih_source()
kermany_src = DirSource(kagglehub.dataset_download(KERMANY_HANDLE))

# RSNA: only the radiologists' label file is needed, the images are the NIH ones.
rsna_classes = read_csv_from(
    kagglehub.competition_download(RSNA_COMPETITION, path="stage_2_detailed_class_info.csv"),
    "stage_2_detailed_class_info.csv")
rsna_boxes = read_csv_from(
    kagglehub.competition_download(RSNA_COMPETITION, path="stage_2_train_labels.csv"),
    "stage_2_train_labels.csv")

response = requests.get(RSNA_MAPPING_URL, timeout=300)
response.raise_for_status()
rsna_map = pd.DataFrame(response.json())

print(f"NIH files        : {len(nih_src.files):,}")
print(f"Kermany files    : {len(kermany_src.files):,}")
print(f"RSNA label rows  : {len(rsna_classes):,} ({rsna_classes.patientId.nunique():,} images)")
print(f"RSNA->NIH mapping: {len(rsna_map):,} entries")
print(f"Done in {time.time() - t0:.0f} s")

## 3. Labels — variant (b)

**NIH + RSNA (adult patients).** NIH labels were mined from radiology reports by NLP and are noisy.
For the 26,684 NIH images re-read by radiologists in the RSNA challenge we use the RSNA label instead:

| Case | Label |
|---|---|
| RSNA *Lung Opacity* (pneumonia-like opacity confirmed by a radiologist) | 1 |
| RSNA *Normal* | 0 |
| RSNA *No Lung Opacity / Not Normal* (other disease, radiologist excluded pneumonia) | 0 if `all_non_pneumonia`, else dropped |
| not in RSNA, NIH label contains *Pneumonia* | 1 |
| not in RSNA, NIH *No Finding* | 0 |
| not in RSNA, NIH *Consolidation* / *Infiltration* without *Pneumonia* | **dropped** — pneumonia-like but never verified |
| not in RSNA, any other NIH finding | 0 if `all_non_pneumonia`, else dropped |

**Kermany (paediatric patients).** Folder name gives Normal / Pneumonia, the file name gives bacterial / viral
(used later for the 3-class task, N1).

In [ ]:
# ---- NIH metadata ----
nih_meta = pd.read_csv(io.BytesIO(nih_src.read(find_file(nih_src, "Data_Entry_2017.csv"))))
nih_meta = nih_meta.rename(columns={"Image Index": "image_id", "Finding Labels": "findings",
                                    "Patient ID": "patient", "View Position": "view",
                                    "Patient Age": "age", "Patient Gender": "sex"})
nih_test_ids = set(nih_src.read(find_file(nih_src, "test_list.txt")).decode().split())
nih_meta["official_split"] = np.where(nih_meta.image_id.isin(nih_test_ids), "test", "train_val")

# ---- RSNA labels, attached to NIH file names through the mapping ----
rsna_cls = rsna_classes.drop_duplicates("patientId")[["patientId", "class"]]  # one row per box -> one per image
rsna_cls = rsna_cls.merge(rsna_map[["subset_img_id", "img_id"]], left_on="patientId",
                          right_on="subset_img_id", how="left")
unmapped = rsna_cls.img_id.isna().sum()
print(f"RSNA images: {len(rsna_cls):,} | not found in the mapping: {unmapped}")
assert unmapped == 0, "every RSNA label must map to exactly one NIH image"
assert rsna_cls.img_id.is_unique

nih_meta = nih_meta.merge(rsna_cls[["img_id", "class"]].rename(columns={"img_id": "image_id", "class": "rsna_class"}),
                          on="image_id", how="left")
print(rsna_cls["class"].value_counts().to_string())

In [ ]:
PNEUMONIA_LIKE = {"Consolidation", "Infiltration"}


def nih_label(findings, rsna_class, negative_policy=NEGATIVE_POLICY):
    # Variant (b) labelling rule. Returns (label or None if dropped, reason).
    keep_other = negative_policy == "all_non_pneumonia"
    if isinstance(rsna_class, str):                       # a radiologist looked at this image
        if rsna_class == "Lung Opacity":
            return 1, "RSNA: lung opacity"
        if rsna_class == "Normal":
            return 0, "RSNA: normal"
        return (0 if keep_other else None), "RSNA: not normal, no opacity"
    labels = set(findings.split("|"))
    if "Pneumonia" in labels:
        return 1, "NIH: pneumonia"
    if labels == {"No Finding"}:
        return 0, "NIH: no finding"
    if labels & PNEUMONIA_LIKE:
        return None, "NIH: unverified consolidation/infiltration"
    return (0 if keep_other else None), "NIH: other finding"


assert nih_label("Pneumonia|Effusion", None) == (1, "NIH: pneumonia")
assert nih_label("Infiltration", None)[0] is None
assert nih_label("Infiltration", "Lung Opacity") == (1, "RSNA: lung opacity")
assert nih_label("Nodule", "No Lung Opacity / Not Normal", "normal_only")[0] is None
assert nih_label("Nodule", None, "all_non_pneumonia") == (0, "NIH: other finding")

nih_meta[["label", "reason"]] = [nih_label(f, r) for f, r in zip(nih_meta.findings, nih_meta.rsna_class)]
reasons = (nih_meta.groupby("reason", dropna=False)
           .agg(images=("image_id", "size"), label=("label", "first"))
           .sort_values("images", ascending=False))
reasons["used"] = reasons.label.notna()
display(reasons)

In [ ]:
# ---- attach image file keys to NIH rows ----
nih_files = {Path(k).name: k for k in nih_src.files if k.endswith(".png")}
nih_meta["file_key"] = nih_meta.image_id.map(nih_files)
missing = nih_meta.file_key.isna().sum()
print(f"NIH images listed in metadata but missing on disk: {missing}")

nih_df = nih_meta[nih_meta.label.notna() & nih_meta.file_key.notna()].copy()
nih_df = pd.DataFrame({
    "image_id": nih_df.image_id, "source": "NIH", "file_key": nih_df.file_key,
    "label": nih_df.label.astype(int), "subtype": np.where(nih_df.label == 1, "pneumonia", "no_pneumonia"),
    "patient_id": "nih_" + nih_df.patient.astype(str), "label_source": np.where(nih_df.rsna_class.notna(), "RSNA", "NIH"),
    "official_split": nih_df.official_split,
})
print(f"NIH images used: {len(nih_df):,}")

In [ ]:
# ---- Kermany ----
def kermany_table(source):
    rows, seen = [], set()
    for key in source.files:
        parts = Path(key).parts
        if "__MACOSX" in parts or not key.lower().endswith((".jpeg", ".jpg", ".png")):
            continue
        split = next((p for p in reversed(parts) if p in ("train", "val", "test")), None)
        cls = next((p for p in reversed(parts) if p in ("NORMAL", "PNEUMONIA")), None)
        name = parts[-1]
        if split is None or cls is None or (split, cls, name) in seen:   # the Kaggle zip contains a duplicate copy
            continue
        seen.add((split, cls, name))
        person = re.match(r"(person\d+)_", name)
        if cls == "PNEUMONIA":
            subtype = "bacterial" if "bacteria" in name.lower() else "viral" if "virus" in name.lower() else "unknown"
        else:
            subtype = "normal"
        rows.append({
            "image_id": f"{split}/{cls}/{name}", "source": "Kermany", "file_key": key,
            "label": int(cls == "PNEUMONIA"), "subtype": subtype,
            # Pneumonia files carry a patient id (person123_bacteria_456.jpeg); normal ones do not (IM-0001-0001.jpeg),
            # so each normal image is its own group = plain stratified split (limitation noted in the proposal).
            # The official val folder is merged into train, so train/val share one id space ("trainval").
            "patient_id": (f"kermany_{'test' if split == 'test' else 'trainval'}_{person.group(1)}" if person
                           else f"kermany_img_{split}_{name}"),
            "label_source": "Kermany",
            "official_split": "test" if split == "test" else "train_val",   # the 16 official val images join train
        })
    return pd.DataFrame(rows)


kermany_df = kermany_table(kermany_src)
print(f"Kermany images: {len(kermany_df):,}")
display(pd.crosstab(kermany_df.official_split, kermany_df.subtype, margins=True))

## 4. Train / validation / test split without leakage (D3)

* **Test sets stay untouched**: the official NIH patient-wise test list and the official Kermany test folder (624 images).
  They are used once, for the final evaluation.
* **Validation** (≈15 %) is carved out of the training pool with `StratifiedGroupKFold`: groups = patients, so one
  patient never appears in two splits, and the pneumonia ratio is kept the same in train and validation.

In [ ]:
def split_train_val(df, val_fraction=VAL_FRACTION, seed=SEED):
    df = df.copy()
    df["split"] = np.where(df.official_split == "test", "test", "train")
    pool = df[df.split == "train"]
    n_splits = max(2, round(1 / val_fraction))
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    _, val_idx = next(sgkf.split(pool, pool.label, groups=pool.patient_id))
    df.loc[pool.index[val_idx], "split"] = "val"
    return df


data = pd.concat([split_train_val(nih_df), split_train_val(kermany_df)], ignore_index=True)

# ---- leakage checks ----
assert data.image_id.is_unique, "duplicate images"
for source, part in data.groupby("source"):
    patients = {s: set(part.loc[part.split == s, "patient_id"]) for s in ("train", "val", "test")}
    for a, b in (("train", "val"), ("train", "test"), ("val", "test")):
        overlap = patients[a] & patients[b]
        assert not overlap, f"{source}: {len(overlap)} patients in both {a} and {b}"
print("No patient appears in more than one split.")

summary = data.pivot_table(index=["source", "split"], columns="label", values="image_id", aggfunc="count",
                           fill_value=0).rename(columns={0: "no pneumonia", 1: "pneumonia"})
summary["total"] = summary.sum(axis=1)
summary["% pneumonia"] = (100 * summary["pneumonia"] / summary["total"]).round(1)
summary = summary.reindex(["train", "val", "test"], level="split")
display(summary)
print(f"TOTAL images: {len(data):,}  ({data.label.sum():,} pneumonia, {(data.label == 0).sum():,} no pneumonia)")

data.drop(columns="file_key").to_csv(WORK_DIR / "pneumodetect_splits.csv", index=False)
print("Split table saved to", WORK_DIR / "pneumodetect_splits.csv")

## 5. Image preprocessing (D2)

Decoding 100k full-size PNGs (1024×1024) every epoch would make the GPU wait for the CPU, so every image is decoded
**once**: converted to grayscale, resized to 224×224 and stored as `uint8` in a memory-mapped array (~5.6 GB).
The input pipeline (section 7) turns it into 3 channels; the backbone-specific normalisation (e.g. ResNet50
`preprocess_input`) is applied inside the model.

In [ ]:
CACHE_FILE = CACHE_DIR / f"images_{IMG_SIZE}.npy"
MANIFEST_FILE = CACHE_DIR / f"manifest_{IMG_SIZE}.csv"
SOURCES = {"NIH": nih_src, "Kermany": kermany_src}


def load_image(source_name, key, size=IMG_SIZE):
    with Image.open(io.BytesIO(SOURCES[source_name].read(key))) as img:
        img = img.convert("L")
        return np.asarray(img.resize((size, size), Image.Resampling.BILINEAR, reducing_gap=2.0), dtype=np.uint8)


def build_cache(df, workers=min(32, (os.cpu_count() or 2) * 4)):
    images = np.lib.format.open_memmap(CACHE_FILE, mode="w+", dtype=np.uint8, shape=(len(df), IMG_SIZE, IMG_SIZE))
    failed = np.zeros(len(df), dtype=bool)

    def work(i):
        try:
            images[i] = load_image(df.source.iat[i], df.file_key.iat[i])
        except Exception as err:          # corrupted file -> report and drop, do not crash after an hour of work
            failed[i] = True
            print("failed:", df.image_id.iat[i], err)

    with ThreadPoolExecutor(workers) as pool:
        list(tqdm(pool.map(work, range(len(df))), total=len(df), desc="Resizing to 224x224"))
    images.flush()
    return failed


data = data.reset_index(drop=True)
cached = (MANIFEST_FILE.exists() and CACHE_FILE.exists()
          and pd.read_csv(MANIFEST_FILE).image_id.tolist() == data.image_id.tolist())
if cached:
    print("Using the existing cache:", CACHE_FILE)
    manifest = pd.read_csv(MANIFEST_FILE)
else:
    t0 = time.time()
    failed = build_cache(data)
    print(f"Cache built in {(time.time() - t0) / 60:.1f} min, failed images: {failed.sum()}")
    manifest = data.drop(columns="file_key").assign(cache_idx=np.arange(len(data)))[~failed].reset_index(drop=True)
    manifest.to_csv(MANIFEST_FILE, index=False)

IMAGES = np.load(CACHE_FILE, mmap_mode="r")
print("Cache:", IMAGES.shape, IMAGES.dtype, f"{IMAGES.nbytes / 1e9:.1f} GB")

if NIH_ACCESS == "zip" and DELETE_NIH_ZIP_AFTER_CACHE and isinstance(nih_src, ZipSource):
    Path(nih_src.zip_path).unlink(missing_ok=True)      # the 42 GB archive is not needed any more
    print("NIH archive removed, disk space freed.")

In [ ]:
# D3, second check: an identical image in train and in val/test would leak the answer into the evaluation.
# Hash every cached image; copies of training images are removed from val/test (and val copies from test).
import hashlib

manifest["hash"] = [hashlib.md5(IMAGES[i].tobytes()).hexdigest() for i in tqdm(manifest.cache_idx, desc="Hashing")]
seen_train = set(manifest.loc[manifest.split == "train", "hash"])
seen_val = set(manifest.loc[manifest.split == "val", "hash"])
leak = (((manifest.split != "train") & manifest.hash.isin(seen_train))
        | ((manifest.split == "test") & manifest.hash.isin(seen_val)))
if leak.any():
    print("Exact duplicates removed from evaluation splits:")
    display(pd.crosstab(manifest.loc[leak, "source"], manifest.loc[leak, "split"]))
else:
    print("No exact duplicate images between train, val and test.")
manifest = manifest[~leak].reset_index(drop=True)
print(f"Images available for modelling: {len(manifest):,}")

## 6. Exploratory data analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
counts = manifest.groupby(["split", "source", "label"]).size().unstack("label", fill_value=0)
counts = counts.reindex(["train", "val", "test"], level="split")
counts.rename(columns={0: "No pneumonia", 1: "Pneumonia"}).plot.bar(stacked=True, ax=axes[0], color=["#4C78A8", "#E45756"])
axes[0].set_title("Figure 1. Images per split and source")
axes[0].set_xlabel("(split, source)")
axes[0].set_ylabel("Number of images")
axes[0].tick_params(axis="x", rotation=30)

sub = manifest[manifest.source == "Kermany"].subtype.value_counts()
nih_reason = manifest[manifest.source == "NIH"].groupby(["label_source", "label"]).size()
nih_reason.index = [f"{s} / {'pneumonia' if l else 'no pneumonia'}" for s, l in nih_reason.index]
pd.concat([nih_reason.rename(lambda s: "NIH: label from " + s), sub.rename(lambda s: "Kermany: " + s)]).plot.barh(
    ax=axes[1], color="#72B7B2")
axes[1].set_title("Figure 2. Where the labels come from")
axes[1].set_xlabel("Number of images")
plt.tight_layout()
plt.show()

In [ ]:
rng = np.random.default_rng(SEED)
groups = [("NIH", 0), ("NIH", 1), ("Kermany", 0), ("Kermany", 1)]
fig, axes = plt.subplots(len(groups), 6, figsize=(13, 9))
for row, (source, label) in enumerate(groups):
    idx = manifest[(manifest.source == source) & (manifest.label == label)].cache_idx.to_numpy()
    for col, i in enumerate(rng.choice(idx, size=min(6, len(idx)), replace=False)):
        axes[row, col].imshow(IMAGES[i], cmap="gray")
    for ax in axes[row]:
        ax.axis("off")
    axes[row, 0].set_title(f"{source} — {'Pneumonia' if label else 'No pneumonia'}", loc="left", fontsize=10)
fig.suptitle("Figure 3. Random training examples after preprocessing (224×224)")
plt.tight_layout()
plt.show()

In [ ]:
# Domain shift check: adult (NIH) and paediatric (Kermany) images differ in brightness/contrast. A CNN could learn
# "which hospital" instead of "which disease" - we keep this in mind when reading the results.
sample = manifest.groupby("source").sample(n=min(2000, manifest.source.value_counts().min()), random_state=SEED)
stats = sample.assign(mean=[IMAGES[i].mean() for i in sample.cache_idx], std=[IMAGES[i].std() for i in sample.cache_idx])
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for source, part in stats.groupby("source"):
    axes[0].hist(part["mean"], bins=40, alpha=0.6, label=source)
    axes[1].hist(part["std"], bins=40, alpha=0.6, label=source)
axes[0].set_title("Figure 4a. Mean pixel intensity per image")
axes[1].set_title("Figure 4b. Pixel standard deviation (contrast) per image")
for ax in axes:
    ax.set_xlabel("Grey level (0-255)")
    ax.set_ylabel("Images")
    ax.legend()
plt.tight_layout()
plt.show()
display(stats.groupby("source")[["mean", "std"]].describe().round(1))

## 7. Input pipeline, augmentation (D7) and class weights (D8)

`tf.data` reads batches from the memory-mapped cache, converts them to 3 channels and prefetches the next batch
while the GPU is busy. Augmentation is a Keras block that will be the first layer of the model, so it runs on the GPU
and only during training: rotation ±10°, shifts up to 10 %, zoom up to 10 %, brightness ±10 %.
**No vertical flips** — an upside-down chest is anatomically unrealistic.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
LABELS_BY_IDX = np.zeros(len(IMAGES), dtype=np.float32)
LABELS_BY_IDX[manifest.cache_idx.to_numpy()] = manifest.label.to_numpy()


def _read_batch(idx):
    idx = np.sort(idx)                      # sorted reads are faster on a memory map; order inside a batch is irrelevant
    return IMAGES[idx], LABELS_BY_IDX[idx]


def _to_model_input(images, labels):
    images = tf.ensure_shape(images, [None, IMG_SIZE, IMG_SIZE])
    images = tf.repeat(tf.cast(images, tf.float32)[..., None], 3, axis=-1)     # grayscale -> 3 channels, range 0..255
    return images, tf.ensure_shape(labels, [None])


def make_dataset(split, training=False, sources=None, batch_size=BATCH_SIZE):
    rows = manifest[manifest.split == split]
    if sources is not None:
        rows = rows[rows.source.isin(sources)]
    ds = tf.data.Dataset.from_tensor_slices(rows.cache_idx.to_numpy())
    if training:
        ds = ds.shuffle(len(rows), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(batch_size)
    ds = ds.map(lambda i: tf.numpy_function(_read_batch, [i], (tf.uint8, tf.float32)), num_parallel_calls=AUTOTUNE)
    return ds.map(_to_model_input, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)


train_ds = make_dataset("train", training=True)
val_ds = make_dataset("val")
test_nih_ds = make_dataset("test", sources=["NIH"])
test_kermany_ds = make_dataset("test", sources=["Kermany"])     # the 624-image benchmark from the proposal

x, y = next(iter(train_ds))
print("Batch:", x.shape, x.dtype, "| labels:", y.numpy()[:10], "| pixel range:", float(tf.reduce_min(x)), "-", float(tf.reduce_max(x)))

In [ ]:
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(10 / 360, fill_mode="constant"),       # ±10°
    tf.keras.layers.RandomTranslation(0.10, 0.10, fill_mode="constant"),  # shifts up to 10 %
    tf.keras.layers.RandomZoom(0.10, fill_mode="constant"),               # zoom up to 10 %
    tf.keras.layers.RandomBrightness(0.10, value_range=(0, 255)),         # brightness ±10 %
], name="augmentation")

example = x[:1]
fig, axes = plt.subplots(1, 6, figsize=(13, 2.6))
axes[0].imshow(example[0, ..., 0].numpy(), cmap="gray", vmin=0, vmax=255)
axes[0].set_title("original")
for ax in axes[1:]:
    ax.imshow(augmentation(example, training=True)[0, ..., 0].numpy(), cmap="gray", vmin=0, vmax=255)
    ax.set_title("augmented")
for ax in axes:
    ax.axis("off")
fig.suptitle("Figure 5. The same X-ray after random augmentation")
plt.tight_layout()
plt.show()

In [ ]:
# D8: "balanced" weights = n_samples / (n_classes * n_class_samples): the rare pneumonia class gets a larger weight in the loss.
train_labels = manifest.loc[manifest.split == "train", "label"].to_numpy()
weights = compute_class_weight("balanced", classes=np.array([0, 1]), y=train_labels)
CLASS_WEIGHT = {0: float(weights[0]), 1: float(weights[1])}
print(f"Training images: {len(train_labels):,} | pneumonia share: {train_labels.mean():.1%}")
print("Class weights:", {k: round(v, 3) for k, v in CLASS_WEIGHT.items()})

## 8. Evaluation protocol (D10)

Every model is evaluated the same way on three sets:

* **val** — used to choose hyperparameters (never the test sets);
* **test NIH** — official patient-wise NIH test list (adults);
* **test Kermany** — official Kermany test folder, 624 children; the success targets of the proposal refer to this set.

Pneumonia is only ≈6 % of the training data, so **accuracy alone is misleading** (predicting "no pneumonia" for every
image is already ≈94 % accurate on NIH). We therefore report sensitivity (recall of pneumonia), specificity, precision,
F1, ROC-AUC and PR-AUC (average precision) next to accuracy.

In [ ]:
from sklearn.metrics import (accuracy_score, average_precision_score, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score, roc_curve, ConfusionMatrixDisplay)

ROW_IDX = manifest.cache_idx.to_numpy()                  # all images, in manifest order
assert np.all(np.diff(ROW_IDX) > 0)
Y_ALL = manifest.label.to_numpy()
EVAL_SETS = {
    "val": (manifest.split == "val").to_numpy(),
    "test NIH": ((manifest.split == "test") & (manifest.source == "NIH")).to_numpy(),
    "test Kermany": ((manifest.split == "test") & (manifest.source == "Kermany")).to_numpy(),
}
TRAIN_MASK = (manifest.split == "train").to_numpy()
RESULTS = []          # one row per (model, evaluation set); shared by baselines and CNNs


def ordered_dataset(batch_size=128):
    # All images in manifest order (no shuffling, no augmentation) - used for feature extraction and prediction.
    ds = tf.data.Dataset.from_tensor_slices(ROW_IDX).batch(batch_size)
    ds = ds.map(lambda i: tf.numpy_function(lambda j: IMAGES[j], [i], tf.uint8), num_parallel_calls=AUTOTUNE)
    ds = ds.map(lambda x: tf.repeat(tf.cast(tf.ensure_shape(x, [None, IMG_SIZE, IMG_SIZE]), tf.float32)[..., None], 3, -1),
                num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)


def metrics(y_true, scores, threshold):
    y_pred = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "sensitivity": recall_score(y_true, y_pred, zero_division=0),
        "specificity": tn / (tn + fp) if tn + fp else np.nan,
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, scores) if len(set(y_true)) > 1 else np.nan,
        "PR-AUC": average_precision_score(y_true, scores) if len(set(y_true)) > 1 else np.nan,
    }


def evaluate(name, scores, threshold):
    # `scores` covers every image in manifest order; higher = more likely pneumonia.
    for set_name, mask in EVAL_SETS.items():
        if not mask.any():
            continue
        RESULTS[:] = [r for r in RESULTS if (r["model"], r["set"]) != (name, set_name)]   # re-running a cell replaces
        RESULTS.append({"model": name, "set": set_name, **metrics(Y_ALL[mask], scores[mask], threshold)})


def results_table(sets=("val", "test NIH", "test Kermany"), columns=("accuracy", "sensitivity", "specificity", "F1", "ROC-AUC")):
    df = pd.DataFrame(RESULTS)
    df = df[df.set.isin(sets)].pivot(index="model", columns="set", values=list(columns))
    df = df.reindex(columns=pd.MultiIndex.from_product([columns, sets]))
    return df.reindex(dict.fromkeys(r["model"] for r in RESULTS)).round(3)

## 9. Classical baselines (D11)

The baselines answer the question *"how much does deep learning add?"*. All of them use `class_weight="balanced"`
(D8) and pick the regularisation strength `C` on the validation ROC-AUC.

| # | Features | Classifier |
|---|---|---|
| 0 | none | majority class (shows why accuracy is misleading) |
| 1–2 | raw pixels: 224×224 averaged down to 56×56 = 3,136 values → standardised → PCA, 256 components | logistic regression, linear SVM |
| 3–4 | 2,048-d features of an ImageNet ResNet50 with **frozen** weights (no training on X-rays) | logistic regression, linear SVM |

Baselines 3–4 isolate the effect of *fine-tuning*: the CNN in section 10 uses the same backbone, but trains it.
Kernel (RBF) SVMs scale quadratically with the number of samples and are not practical for ~85k training images,
hence linear models.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

dummy = DummyClassifier(strategy="most_frequent").fit(np.zeros((TRAIN_MASK.sum(), 1)), Y_ALL[TRAIN_MASK])
evaluate("0 Majority class", dummy.predict(np.zeros((len(Y_ALL), 1))).astype(float), threshold=0.5)
display(results_table(columns=("accuracy", "sensitivity", "specificity", "F1")).loc[["0 Majority class"]])

In [ ]:
def fit_and_select(name, make_clf, Cs, X):
    # Fit one classifier per C on train, keep the best by validation ROC-AUC, evaluate it on all sets.
    Xtr, ytr = X[TRAIN_MASK], Y_ALL[TRAIN_MASK]
    val = EVAL_SETS["val"]
    best, log = None, []
    for C in Cs:
        t0 = time.time()
        clf = make_clf(C).fit(Xtr, ytr)
        auc = roc_auc_score(Y_ALL[val], clf.decision_function(X[val]))
        log.append({"C": C, "val ROC-AUC": round(auc, 4), "fit time, s": round(time.time() - t0, 1)})
        if best is None or auc > best[0]:
            best = (auc, C, clf)
    print(f"{name}: best C = {best[1]}")
    display(pd.DataFrame(log).set_index("C").T)
    scores = best[2].decision_function(X)          # signed distance / logit: 0 is the decision boundary
    evaluate(name, scores, threshold=0.0)
    return best[2], scores


make_logreg = lambda C: LogisticRegression(C=C, class_weight="balanced", max_iter=3000)
make_svm = lambda C: LinearSVC(C=C, class_weight="balanced", max_iter=5000)
BASELINE_SCORES = {}

### 9.1 Pixel features

In [ ]:
PIX = 56
assert IMG_SIZE % PIX == 0
factor = IMG_SIZE // PIX
X_pix = np.empty((len(ROW_IDX), PIX * PIX), dtype=np.float32)
for s in tqdm(range(0, len(ROW_IDX), 2048), desc="Pixel features"):
    block = IMAGES[ROW_IDX[s:s + 2048]].astype(np.float32) / 255.0
    X_pix[s:s + 2048] = block.reshape(-1, PIX, factor, PIX, factor).mean(axis=(2, 4)).reshape(len(block), -1)

n_comp = min(256, TRAIN_MASK.sum() - 1, X_pix.shape[1])
reducer = make_pipeline(StandardScaler(), PCA(n_components=n_comp, svd_solver="randomized", random_state=SEED))
reducer.fit(X_pix[TRAIN_MASK])                      # fitted on training images only (no information from val/test)
Z_pix = reducer.transform(X_pix).astype(np.float32)
print(f"{X_pix.shape[1]} pixels -> {n_comp} PCA components, explained variance "
      f"{reducer[-1].explained_variance_ratio_.sum():.1%}")

_, BASELINE_SCORES["1 Pixels + LogReg"] = fit_and_select("1 Pixels + LogReg", make_logreg, [0.001, 0.01, 0.1, 1.0], Z_pix)
_, BASELINE_SCORES["2 Pixels + Linear SVM"] = fit_and_select("2 Pixels + Linear SVM", make_svm, [0.0001, 0.001, 0.01, 0.1], Z_pix)

### 9.2 Frozen ResNet50 features

In [ ]:
FEAT_FILE = CACHE_DIR / f"resnet50_imagenet_avgpool_{IMG_SIZE}.npy"
if FEAT_FILE.exists() and np.load(FEAT_FILE, mmap_mode="r").shape[0] == len(ROW_IDX):
    X_cnn = np.load(FEAT_FILE)
    print("Loaded cached features:", X_cnn.shape)
else:
    if not tf.config.list_physical_devices("GPU"):
        print("WARNING: no GPU - extracting features for ~100k images on a CPU takes hours.")
    feature_extractor = tf.keras.Sequential([
        tf.keras.Input((IMG_SIZE, IMG_SIZE, 3)),
        tf.keras.layers.Lambda(tf.keras.applications.resnet50.preprocess_input),
        tf.keras.applications.ResNet50(weights="imagenet", include_top=False, pooling="avg"),
    ], name="frozen_resnet50")
    t0 = time.time()
    X_cnn = feature_extractor.predict(ordered_dataset(), verbose=1).astype(np.float32)
    np.save(FEAT_FILE, X_cnn)
    print(f"Features {X_cnn.shape} extracted in {(time.time() - t0) / 60:.1f} min")

scaler = StandardScaler().fit(X_cnn[TRAIN_MASK])
Z_cnn = scaler.transform(X_cnn).astype(np.float32)
_, BASELINE_SCORES["3 ResNet50 features + LogReg"] = fit_and_select(
    "3 ResNet50 features + LogReg", make_logreg, [0.0001, 0.001, 0.01, 0.1], Z_cnn)
_, BASELINE_SCORES["4 ResNet50 features + Linear SVM"] = fit_and_select(
    "4 ResNet50 features + Linear SVM", make_svm, [0.00001, 0.0001, 0.001, 0.01], Z_cnn)

### 9.3 Baseline results

In [ ]:
print("Table 1. Baselines on the validation and both test sets (threshold = default decision boundary)")
display(results_table())
pd.DataFrame(RESULTS).to_csv(WORK_DIR / "results.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, set_name in zip(axes, ["test NIH", "test Kermany"]):
    mask = EVAL_SETS[set_name]
    for name, scores in BASELINE_SCORES.items():
        fpr, tpr, _ = roc_curve(Y_ALL[mask], scores[mask])
        ax.plot(fpr, tpr, label=f"{name} (AUC {roc_auc_score(Y_ALL[mask], scores[mask]):.3f})")
    ax.plot([0, 1], [0, 1], "k--", lw=1, label="chance")
    ax.set_title(f"Figure 6{'ab'[set_name == 'test Kermany']}. ROC curves, {set_name}")
    ax.set_xlabel("False positive rate (1 − specificity)")
    ax.set_ylabel("True positive rate (sensitivity)")
    ax.legend(fontsize=8, loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
val_auc = {r["model"]: r["ROC-AUC"] for r in RESULTS if r["set"] == "val" and r["model"] in BASELINE_SCORES}
BEST_BASELINE = max(val_auc, key=val_auc.get)
print("Best baseline by validation ROC-AUC:", BEST_BASELINE)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, set_name in zip(axes, ["test NIH", "test Kermany"]):
    mask = EVAL_SETS[set_name]
    ConfusionMatrixDisplay.from_predictions(
        Y_ALL[mask], (BASELINE_SCORES[BEST_BASELINE][mask] >= 0).astype(int),
        display_labels=["No pneumonia", "Pneumonia"], cmap="Blues", colorbar=False, ax=ax)
    ax.set_title(set_name)
fig.suptitle(f"Figure 7. Confusion matrices of the best baseline ({BEST_BASELINE})")
plt.tight_layout()
plt.show()

**Next sections** (to be added): ResNet50 / VGG16 transfer learning with two-stage fine-tuning and callbacks
(D5, D6, D9); comparison with the baselines on the same sets (D10); nice-to-do items N1–N8.